# Module 6 – GraphRAG: When Regular RAG Isn't Enough

## 🎯 What You'll Learn (In Plain English)

Imagine you have a **library of technical documents** about a software system. You want to ask:

> "What happens if the Authentication Service goes down?"

**Regular RAG** (what we built in Modules 1-5) would:
1. Search for chunks containing "Authentication Service"
2. Return the top 5 most similar chunks
3. Ask the LLM to answer based on those chunks

**The Problem**: Regular RAG can't "connect the dots" across documents. It doesn't know that:
- The API Gateway **depends on** Authentication Service
- The User Portal **depends on** API Gateway
- So if Auth goes down → API Gateway fails → User Portal fails!

**GraphRAG solves this** by building a **knowledge graph** of entities and relationships.

---

## 🧠 GraphRAG in One Picture

```
┌─────────────────────────────────────────────────────────────────┐
│                     REGULAR RAG (Module 5)                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Documents → Chunks → Vectors → Search → Top-K → Answer       │
│                                                                 │
│   ✅ Good for: "What is the voltage rating of motor X?"        │
│   ❌ Bad for:  "What depends on motor X?"                      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                     GRAPHRAG (This Module!)                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Documents → Entities → Relationships → Graph → Traverse      │
│                                                                 │
│   [Auth Service] ──DEPENDS_ON──► [API Gateway]                  │
│         │                              │                        │
│         └──USED_BY──► [User Portal] ◄──┘                        │
│                                                                 │
│   ✅ Good for: "What depends on Auth Service?"                 │
│   ✅ Good for: "Summarize the entire system architecture"      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 📍 Where We Are in the Workshop

```
Module 0: Setup           ✅ Done
Module 1: Naive RAG       ✅ Done (saw why simple RAG fails)
Module 2: Doc Intelligence✅ Done (extracted structure)
Module 3: Content Under.  ✅ Done (got AI descriptions)
Module 4: Chunking        ✅ Done (created smart chunks)
Module 5: Search          ✅ Done (indexed & retrieved)
Module 6: GraphRAG        👈 YOU ARE HERE (cross-document reasoning)
```

---

## 🔧 What is Microsoft GraphRAG?

**Microsoft GraphRAG** is a free, open-source Python library that:

1. **Reads your documents** (plain text files)
2. **Extracts entities** using an LLM ("API Gateway", "Auth Service", etc.)
3. **Extracts relationships** using an LLM ("depends on", "connects to", etc.)
4. **Builds a knowledge graph** (like a web of connected concepts)
5. **Detects communities** (groups of related entities)
6. **Answers questions** by traversing the graph

### ⚠️ Important: GraphRAG is NOT an Azure Service!

| What | Technology |
|------|------------|
| GraphRAG | Open-source Python library (`pip install graphrag`) |
| LLM for extraction | Azure OpenAI GPT-4.1 (you pay per token) |
| Storage | Local files (Parquet/JSON) - no database needed! |

**GraphRAG uses Azure OpenAI heavily** - expect significant token usage during indexing!

---

# Part 0: Setup & Installation

## 0.1 Install GraphRAG

In [ ]:
# Install Microsoft GraphRAG
# Note: Requires Python >=3.11, <3.14
%pip install graphrag>=2.7.0 --quiet

## 0.2 Load Environment Variables

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
env_path = Path("../../.env")
if env_path.exists():
    load_dotenv(env_path)
    print("✅ Loaded .env file")
else:
    print("⚠️ No .env file found. Make sure environment variables are set.")

# Verify required variables
required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
]

missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    print(f"❌ Missing environment variables: {missing}")
else:
    print("✅ All required environment variables are set")
    print(f"   Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')[:50]}...")

## 0.3 Create GraphRAG Project Structure

GraphRAG needs a specific folder structure:

```
graphrag-demo/
├── input/              ← Your text documents go here
│   └── *.txt
├── output/             ← GraphRAG creates this (entities, relationships, etc.)
├── settings.yaml       ← Configuration file
└── .env                ← API keys
```

In [ ]:
import shutil

# Create GraphRAG project directory
GRAPHRAG_ROOT = Path("./graphrag-demo")
INPUT_DIR = GRAPHRAG_ROOT / "input"

# Clean up if exists (for fresh start)
if GRAPHRAG_ROOT.exists():
    shutil.rmtree(GRAPHRAG_ROOT)
    print("🗑️ Cleaned up existing graphrag-demo folder")

# Create directories
INPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Created: {GRAPHRAG_ROOT}")
print(f"✅ Created: {INPUT_DIR}")

---

# Part 1: Understanding the Data

## 1.1 Create Sample Documents

For this demo, we'll create a few documents about a **fictional software system**.

These documents describe:
- **Services** (Auth Service, API Gateway, User Portal)
- **Dependencies** (what depends on what)
- **Teams** (who owns what)

This is exactly the kind of content where **regular RAG fails** but **GraphRAG shines**!

In [ ]:
# Document 1: System Overview
doc1 = """
# Contoso Platform Architecture Overview

The Contoso Platform is a modern microservices-based system designed for high availability 
and scalability. The platform consists of several core services that work together to 
provide a seamless user experience.

## Core Services

The platform includes the following core services:

1. **Authentication Service (AuthService)**: Handles all user authentication, token 
   generation, and session management. This is the most critical service as all other 
   services depend on it for user verification.

2. **API Gateway**: Acts as the single entry point for all client requests. It routes 
   requests to appropriate backend services and handles rate limiting. The API Gateway 
   depends on AuthService for token validation.

3. **User Portal**: The main web application that end-users interact with. It connects 
   to the API Gateway for all backend operations.

4. **Notification Service**: Sends emails, SMS, and push notifications to users. It 
   operates independently but uses AuthService to verify user preferences.

5. **Data Analytics Service**: Collects and processes usage metrics. It connects to 
   the Message Queue for event streaming.

## System Dependencies

The dependency chain is critical for understanding failure scenarios:
- If AuthService goes down, the API Gateway cannot validate tokens
- If API Gateway fails, the User Portal cannot reach backend services
- The Notification Service can operate in degraded mode without AuthService

## Team Ownership

The Platform Team owns AuthService and API Gateway.
The Frontend Team owns the User Portal.
The Communications Team owns the Notification Service.
The Data Team owns the Data Analytics Service.
"""

(INPUT_DIR / "01_system_overview.txt").write_text(doc1)
print("✅ Created: 01_system_overview.txt")

In [ ]:
# Document 2: Authentication Service Details
doc2 = """
# Authentication Service (AuthService) Technical Specification

## Overview

AuthService is the central authentication and authorization service for the Contoso 
Platform. It implements OAuth 2.0 and OpenID Connect protocols.

## Dependencies

AuthService depends on the following components:
- **PostgreSQL Database**: Stores user credentials and session data
- **Redis Cache**: Caches active sessions for fast token validation
- **Azure Key Vault**: Stores encryption keys and secrets

## Endpoints

- POST /auth/login - User login
- POST /auth/logout - User logout  
- POST /auth/refresh - Refresh access token
- GET /auth/validate - Validate token (used by API Gateway)

## Failure Modes

If AuthService becomes unavailable:
1. API Gateway will reject all requests (cannot validate tokens)
2. User Portal will show "Service Unavailable" errors
3. New user logins will fail
4. Existing sessions with valid cached tokens may continue briefly

## Recovery Procedures

AuthService recovery requires:
1. Verify PostgreSQL connectivity
2. Verify Redis connectivity
3. Check Azure Key Vault access
4. Restart AuthService pods in Kubernetes
5. Verify health endpoint responds with 200 OK

## Team Contact

Primary: Platform Team (platform@contoso.com)
On-call: Platform-Oncall rotation
Escalation: John Smith (Engineering Director)
"""

(INPUT_DIR / "02_auth_service.txt").write_text(doc2)
print("✅ Created: 02_auth_service.txt")

In [ ]:
# Document 3: API Gateway Details
doc3 = """
# API Gateway Technical Specification

## Overview

The API Gateway is the single entry point for all external traffic to the Contoso 
Platform. It provides routing, rate limiting, and authentication verification.

## Dependencies

API Gateway depends on:
- **AuthService**: For token validation (critical dependency)
- **Redis Cache**: For rate limiting counters
- **Service Registry**: For dynamic backend service discovery

## Routing Rules

| Path Pattern | Backend Service | Auth Required |
|-------------|-----------------|---------------|
| /api/users/* | User Service | Yes |
| /api/products/* | Product Service | Yes |
| /api/notifications/* | Notification Service | Yes |
| /api/public/* | Public Service | No |
| /health | Self | No |

## Failure Scenarios

### Scenario 1: AuthService Unavailable
Impact: All authenticated requests fail with 503 Service Unavailable
Affected Services: User Portal, Mobile App, Third-party integrations
Mitigation: Circuit breaker activates after 5 consecutive failures

### Scenario 2: Backend Service Unavailable
Impact: Requests to specific endpoints fail
Mitigation: Returns cached response if available, otherwise 503

## Performance Requirements

- Latency: p99 < 50ms (excluding backend time)
- Throughput: 10,000 requests/second
- Availability: 99.99% uptime

## Team Contact

Primary: Platform Team (platform@contoso.com)
Architecture: Sarah Johnson (Principal Architect)
"""

(INPUT_DIR / "03_api_gateway.txt").write_text(doc3)
print("✅ Created: 03_api_gateway.txt")

In [ ]:
# Document 4: Incident Report
doc4 = """
# Incident Report: Platform Outage - January 15, 2026

## Incident Summary

On January 15, 2026 at 14:32 UTC, the Contoso Platform experienced a complete outage 
lasting 47 minutes. All user-facing services were unavailable.

## Timeline

- 14:32 UTC: AuthService health checks start failing
- 14:33 UTC: API Gateway begins returning 503 errors
- 14:34 UTC: User Portal shows "Service Unavailable" to all users
- 14:35 UTC: PagerDuty alert triggers, Platform Team engaged
- 14:42 UTC: Root cause identified - PostgreSQL connection pool exhausted
- 14:55 UTC: PostgreSQL connection limit increased
- 15:02 UTC: AuthService recovers
- 15:05 UTC: API Gateway health restored
- 15:08 UTC: User Portal fully operational
- 15:19 UTC: All systems confirmed stable

## Root Cause

The PostgreSQL database reached its maximum connection limit (100) due to a connection 
leak in AuthService version 2.3.1. This prevented AuthService from validating tokens, 
which cascaded to API Gateway and User Portal failures.

## Impact

- 47 minutes of complete platform unavailability
- Approximately 125,000 users affected
- Estimated revenue impact: $45,000
- 342 support tickets created

## Action Items

1. [Platform Team] Fix connection leak in AuthService - COMPLETED
2. [Platform Team] Increase PostgreSQL connection limit to 500
3. [Platform Team] Add connection pool monitoring alerts
4. [Data Team] Improve cascade failure detection
5. [All Teams] Update runbooks with this failure scenario

## Lessons Learned

The strong dependency chain (AuthService → API Gateway → User Portal) means that 
AuthService failures have maximum blast radius. We need to consider:
- Caching valid tokens longer at API Gateway
- Implementing graceful degradation for non-critical features
- Adding redundancy to AuthService
"""

(INPUT_DIR / "04_incident_report.txt").write_text(doc4)
print("✅ Created: 04_incident_report.txt")

In [ ]:
# Document 5: Team Directory
doc5 = """
# Contoso Platform Team Directory

## Platform Team

The Platform Team is responsible for core infrastructure services.

**Services Owned:**
- AuthService (Authentication Service)
- API Gateway
- Service Registry
- Configuration Service

**Team Members:**
- John Smith (Engineering Director) - john.smith@contoso.com
- Sarah Johnson (Principal Architect) - sarah.johnson@contoso.com
- Michael Chen (Senior Engineer) - michael.chen@contoso.com
- Emily Davis (Engineer) - emily.davis@contoso.com

## Frontend Team

The Frontend Team builds and maintains user-facing applications.

**Services Owned:**
- User Portal
- Mobile App
- Admin Dashboard

**Team Members:**
- Lisa Wang (Engineering Manager) - lisa.wang@contoso.com
- David Kim (Lead Developer) - david.kim@contoso.com
- Anna Martinez (UX Designer) - anna.martinez@contoso.com

## Data Team

The Data Team handles analytics and data processing.

**Services Owned:**
- Data Analytics Service
- ETL Pipeline
- Reporting Dashboard

**Team Members:**
- Robert Taylor (Data Engineering Lead) - robert.taylor@contoso.com
- Jennifer Lee (Data Engineer) - jennifer.lee@contoso.com

## Communications Team

The Communications Team manages all notification systems.

**Services Owned:**
- Notification Service
- Email Gateway
- SMS Gateway

**Team Members:**
- Chris Brown (Team Lead) - chris.brown@contoso.com
- Amanda Wilson (Engineer) - amanda.wilson@contoso.com
"""

(INPUT_DIR / "05_team_directory.txt").write_text(doc5)
print("✅ Created: 05_team_directory.txt")

In [ ]:
# Verify all documents
print("\n📁 Documents in input folder:")
print("=" * 50)
for f in sorted(INPUT_DIR.glob("*.txt")):
    size = f.stat().st_size
    print(f"  📄 {f.name} ({size:,} bytes)")

total_size = sum(f.stat().st_size for f in INPUT_DIR.glob("*.txt"))
print(f"\n📊 Total: {len(list(INPUT_DIR.glob('*.txt')))} documents, {total_size:,} bytes")

---

# Part 2: Configure GraphRAG

## 2.1 Understanding the Configuration

GraphRAG needs a `settings.yaml` file that tells it:

1. **Which LLM to use** (Azure OpenAI GPT-4.1)
2. **Which embedding model to use** (text-embedding-3-large)
3. **How to chunk documents**
4. **How to extract entities**

Let's create this configuration:

In [ ]:
# Get Azure OpenAI configuration
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY", "")

# Extract just the resource name from the endpoint
# e.g., "https://myresource.openai.azure.com/" -> "myresource"
import re
match = re.search(r'https://([^.]+)\.', azure_endpoint)
resource_name = match.group(1) if match else "your-resource"

print(f"Azure OpenAI Resource: {resource_name}")
print(f"Endpoint: {azure_endpoint}")

In [ ]:
# Initialize GraphRAG project and create settings for Azure OpenAI
import subprocess
import sys

# First, let GraphRAG create its own default config (may fail if already exists - that's OK)
print("📝 Initializing GraphRAG project structure...")
result = subprocess.run(
    [sys.executable, "-m", "graphrag", "init", "--root", str(GRAPHRAG_ROOT)],
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("✅ GraphRAG init completed")
else:
    print("⚠️ GraphRAG init skipped (folder already exists)")

# Get Azure endpoint and API key from environment
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY", "")

if not azure_endpoint:
    print("❌ ERROR: AZURE_OPENAI_ENDPOINT not set in .env!")
else:
    print(f"✅ Using Azure endpoint: {azure_endpoint}")

# Create settings.yaml for Azure OpenAI with API Key authentication
# Note: GraphRAG 2.7.x uses 'auth_type' field with 'api_key' value
settings_yaml = f"""# GraphRAG Configuration for Azure OpenAI
# For GraphRAG version 2.7.x

models:
  default_chat_model:
    type: azure_openai_chat
    model: gpt-4.1
    api_base: {azure_endpoint}
    api_version: "2024-12-01-preview"
    deployment_name: gpt-4.1
    api_key: ${{GRAPHRAG_API_KEY}}
    auth_type: api_key
    requests_per_minute: 60
    tokens_per_minute: 80000
    max_tokens: 4000
    temperature: 0

  default_embedding_model:
    type: azure_openai_embedding
    model: text-embedding-3-large
    api_base: {azure_endpoint}
    api_version: "2024-12-01-preview"
    deployment_name: text-embedding-3-large
    api_key: ${{GRAPHRAG_API_KEY}}
    auth_type: api_key
    requests_per_minute: 120
    tokens_per_minute: 120000

input:
  type: file
  file_type: text
  base_dir: input

chunking:
  type: tokens
  size: 1200
  overlap: 100

extract_graph:
  entity_types:
    - SERVICE
    - TEAM
    - PERSON
    - TECHNOLOGY
    - INCIDENT
  max_gleanings: 1

cluster_graph:
  max_cluster_size: 10

community_reports:
  max_length: 2000
  max_input_length: 8000

output:
  type: file
  base_dir: output

reporting:
  type: file
  base_dir: output/reports

snapshots:
  graphml: true
"""

# Write settings file
(GRAPHRAG_ROOT / "settings.yaml").write_text(settings_yaml.strip())
print("\n✅ Created: settings.yaml")
print(f"   Using endpoint: {azure_endpoint}")
print("   Auth type: api_key (uses GRAPHRAG_API_KEY from .env)")

In [ ]:
# Create .env file for GraphRAG with API key
# GraphRAG reads GRAPHRAG_API_KEY from this .env file
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY", "")

if not azure_api_key:
    print("❌ ERROR: AZURE_OPENAI_API_KEY not found in .env!")
    print("   Please add your Azure OpenAI API key to the .env file")
else:
    env_content = f"""GRAPHRAG_API_KEY={azure_api_key}
GRAPHRAG_API_BASE={azure_endpoint}
"""
    (GRAPHRAG_ROOT / ".env").write_text(env_content)
    print("✅ Created: graphrag-demo/.env")
    print(f"   API Base: {azure_endpoint[:50]}...")
    print(f"   API Key: {'*' * 20}... (hidden)")

## 2.2 Understanding Entity Types

In our configuration, we told GraphRAG to look for these entity types:

| Entity Type | Examples | Why It Matters |
|-------------|----------|----------------|
| SERVICE | AuthService, API Gateway | Core components to track dependencies |
| TEAM | Platform Team, Data Team | Who owns what |
| PERSON | John Smith, Sarah Johnson | Points of contact |
| TECHNOLOGY | PostgreSQL, Redis | Infrastructure dependencies |
| INCIDENT | January 15 Outage | Historical events |
| ENDPOINT | /auth/login | API surface |

**Pro Tip**: Customize entity types for YOUR domain!

---

# Part 3: Run GraphRAG Indexing

## 3.1 What Happens During Indexing?

This is the **expensive** part (lots of LLM calls!):

```
┌─────────────────────────────────────────────────────────────────┐
│                    GRAPHRAG INDEXING PIPELINE                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Step 1: Read Documents                                         │
│    └── Load all .txt files from input/                          │
│                                                                 │
│  Step 2: Chunk Documents                                        │
│    └── Split into ~1200 character chunks                        │
│                                                                 │
│  Step 3: Extract Entities (LLM 🤖💰)                            │
│    └── "Find all SERVICEs, TEAMs, PERSONs in this text..."     │
│                                                                 │
│  Step 4: Extract Relationships (LLM 🤖💰)                       │
│    └── "How does AuthService relate to API Gateway?"           │
│                                                                 │
│  Step 5: Build Graph                                            │
│    └── Create entity nodes and relationship edges               │
│                                                                 │
│  Step 6: Detect Communities (Leiden Algorithm)                  │
│    └── Group related entities into clusters                     │
│                                                                 │
│  Step 7: Generate Community Summaries (LLM 🤖💰)               │
│    └── "Summarize what this group of entities is about..."     │
│                                                                 │
│  Step 8: Create Embeddings (Embedding Model 💰)                 │
│    └── Vector representations for search                        │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

💰 = Costs money (Azure OpenAI API calls)
```

**⚠️ Warning**: This will use significant tokens! For our small demo (~10KB of text), expect:
- ~50,000-100,000 input tokens
- ~10,000-20,000 output tokens
- Cost: ~$0.50-$2.00 (depending on your Azure pricing)

In [ ]:
# Show what we're about to index
print("📊 Pre-Indexing Summary")
print("=" * 50)
print(f"Documents: {len(list(INPUT_DIR.glob('*.txt')))}")
print(f"Total size: {sum(f.stat().st_size for f in INPUT_DIR.glob('*.txt')):,} bytes")
print(f"\nConfiguration:")
print(f"  - LLM: Azure OpenAI GPT-4.1")
print(f"  - Embeddings: text-embedding-3-large")
print(f"  - Chunk size: 1200 characters")
print(f"\n⚠️ This will make many LLM API calls!")
print(f"   Estimated cost: $0.50-$2.00")

## 3.2 Run the Indexing Pipeline

Now let's run GraphRAG indexing. This will take a few minutes.

**Note**: We're using the Python API instead of the CLI for better notebook integration.

In [ ]:
import subprocess
import os

# Change to the graphrag directory
original_dir = os.getcwd()
os.chdir(GRAPHRAG_ROOT)

print("🚀 Starting GraphRAG Indexing...")
print("   This will take 2-5 minutes depending on your Azure OpenAI quota.")
print("   Watch for progress updates below...\n")
print("=" * 60)

In [ ]:
# Run GraphRAG indexing using Python API (better for notebooks)
import asyncio
from pathlib import Path

try:
    from graphrag.index.run.run import run_pipeline_with_config
    from graphrag.config.create_graphrag_config import create_graphrag_config
    
    print("🚀 Starting GraphRAG Indexing via Python API...")
    print("   This will take 2-5 minutes depending on your Azure OpenAI quota.\n")
    print("=" * 60)
    
    # Load config and run
    config = create_graphrag_config(root_dir=Path("."))
    
    async def run_index():
        await run_pipeline_with_config(config)
    
    # Run the indexing
    await run_index()
    
    print("\n" + "=" * 60)
    print("✅ GraphRAG Indexing Complete!")
    
except ImportError as e:
    print(f"⚠️ Import error: {e}")
    print("\nFalling back to CLI method...")
    
    # Fallback to CLI
    import subprocess
    import sys
    
    # Use the Python executable that has graphrag installed
    python_exe = sys.executable
    
    result = subprocess.run(
        [python_exe, "-m", "graphrag", "index", "--root", "."],
        capture_output=True,
        text=True,
        timeout=600
    )
    
    print(result.stdout)
    if result.stderr:
        print("\n⚠️ Warnings/Errors:")
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n" + "=" * 60)
        print("✅ GraphRAG Indexing Complete!")
    else:
        print(f"\n❌ Indexing failed with return code: {result.returncode}")

except Exception as e:
    print(f"❌ Error during indexing: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Return to original directory
os.chdir(original_dir)
print(f"📁 Returned to: {os.getcwd()}")

---

# Part 4: Explore the Knowledge Graph

## 4.1 What Did GraphRAG Create?

After indexing, GraphRAG creates several output files:

In [ ]:
# List output files
OUTPUT_DIR = GRAPHRAG_ROOT / "output"

print("📁 GraphRAG Output Files:")
print("=" * 60)

if OUTPUT_DIR.exists():
    for item in sorted(OUTPUT_DIR.rglob("*")):
        if item.is_file():
            rel_path = item.relative_to(OUTPUT_DIR)
            size = item.stat().st_size
            print(f"  📄 {rel_path} ({size:,} bytes)")
else:
    print("  ⚠️ Output directory not found. Indexing may not have completed.")

## 4.2 Load and Explore Entities

Let's see what entities GraphRAG extracted!

In [ ]:
import pandas as pd

# Find the entities file (it's usually in a timestamped folder)
entities_files = list(OUTPUT_DIR.rglob("*entities*.parquet"))

if entities_files:
    entities_df = pd.read_parquet(entities_files[0])
    print(f"✅ Loaded {len(entities_df)} entities from {entities_files[0].name}")
    print(f"\nColumns: {list(entities_df.columns)}")
else:
    print("⚠️ No entities file found")
    entities_df = None

In [ ]:
# Display entities
if entities_df is not None:
    print("\n🔍 Extracted Entities:")
    print("=" * 60)
    
    # Show key columns
    display_cols = ['title', 'type', 'description'] if 'type' in entities_df.columns else ['title', 'description']
    available_cols = [c for c in display_cols if c in entities_df.columns]
    
    if available_cols:
        display_df = entities_df[available_cols].head(20)
        # Truncate long descriptions
        if 'description' in display_df.columns:
            display_df['description'] = display_df['description'].str[:100] + '...'
        print(display_df.to_string())
    else:
        print(entities_df.head(20))

In [ ]:
# Count entities by type
if entities_df is not None and 'type' in entities_df.columns:
    print("\n📊 Entity Count by Type:")
    print("=" * 40)
    print(entities_df['type'].value_counts().to_string())

## 4.3 Load and Explore Relationships

Now let's see how entities are connected!

In [ ]:
# Find the relationships file
relationships_files = list(OUTPUT_DIR.rglob("*relationships*.parquet"))

if relationships_files:
    relationships_df = pd.read_parquet(relationships_files[0])
    print(f"✅ Loaded {len(relationships_df)} relationships from {relationships_files[0].name}")
    print(f"\nColumns: {list(relationships_df.columns)}")
else:
    print("⚠️ No relationships file found")
    relationships_df = None

In [ ]:
# Display relationships
if relationships_df is not None:
    print("\n🔗 Extracted Relationships:")
    print("=" * 60)
    
    display_cols = ['source', 'target', 'description']
    available_cols = [c for c in display_cols if c in relationships_df.columns]
    
    if available_cols:
        display_df = relationships_df[available_cols].head(20)
        if 'description' in display_df.columns:
            display_df['description'] = display_df['description'].str[:80] + '...'
        print(display_df.to_string())
    else:
        print(relationships_df.head(20))

## 4.4 Visualize the Knowledge Graph

Let's create a visual representation of the entity relationships!

In [ ]:
# Install visualization library if needed
%pip install pyvis networkx --quiet

In [ ]:
import networkx as nx
from pyvis.network import Network
import json

# Create a NetworkX graph from our data
if entities_df is not None and relationships_df is not None:
    G = nx.Graph()
    
    # Add nodes (entities)
    for _, row in entities_df.iterrows():
        entity_type = row.get('type', 'UNKNOWN')
        G.add_node(
            row['title'],
            title=row.get('description', '')[:200],
            group=entity_type
        )
    
    # Add edges (relationships)
    for _, row in relationships_df.iterrows():
        if row['source'] in G.nodes and row['target'] in G.nodes:
            G.add_edge(
                row['source'],
                row['target'],
                title=row.get('description', '')[:100]
            )
    
    print(f"✅ Created graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
else:
    print("⚠️ Cannot create graph - missing data")
    G = None

In [ ]:
# Create interactive visualization
if G is not None and G.number_of_nodes() > 0:
    # Create PyVis network
    net = Network(notebook=True, height="600px", width="100%", bgcolor="#222222", font_color="white")
    
    # Color mapping for entity types
    colors = {
        'SERVICE': '#4CAF50',      # Green
        'TEAM': '#2196F3',         # Blue
        'PERSON': '#FF9800',       # Orange
        'TECHNOLOGY': '#9C27B0',   # Purple
        'INCIDENT': '#F44336',     # Red
        'ENDPOINT': '#00BCD4',     # Cyan
        'UNKNOWN': '#9E9E9E'       # Gray
    }
    
    # Add nodes with colors
    for node in G.nodes(data=True):
        node_id = node[0]
        node_data = node[1]
        group = node_data.get('group', 'UNKNOWN')
        color = colors.get(group, colors['UNKNOWN'])
        net.add_node(node_id, label=node_id, color=color, title=node_data.get('title', ''))
    
    # Add edges
    for edge in G.edges(data=True):
        net.add_edge(edge[0], edge[1], title=edge[2].get('title', ''))
    
    # Configure physics
    net.set_options("""
    var options = {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -50,
          "centralGravity": 0.01,
          "springLength": 100
        },
        "solver": "forceAtlas2Based"
      }
    }
    """)
    
    # Save and display
    graph_path = GRAPHRAG_ROOT / "knowledge_graph.html"
    net.save_graph(str(graph_path))
    print(f"✅ Graph saved to: {graph_path}")
    print("\n🎨 Color Legend:")
    for entity_type, color in colors.items():
        print(f"   {color} = {entity_type}")
    
    # Display in notebook
    net.show(str(graph_path))
else:
    print("⚠️ No graph data to visualize")

## 4.5 Explore Communities

GraphRAG groups related entities into **communities** and creates summaries for each.

This is what enables **global queries** like "Summarize the platform architecture".

In [ ]:
# Find community reports
community_files = list(OUTPUT_DIR.rglob("*community*.parquet"))

if community_files:
    communities_df = pd.read_parquet(community_files[0])
    print(f"✅ Loaded {len(communities_df)} communities")
    print(f"\nColumns: {list(communities_df.columns)}")
else:
    print("⚠️ No community reports found")
    communities_df = None

In [ ]:
# Display community summaries
if communities_df is not None:
    print("\n🏘️ Community Summaries:")
    print("=" * 60)
    
    for i, row in communities_df.head(5).iterrows():
        print(f"\n📌 Community {i+1}:")
        if 'title' in row:
            print(f"   Title: {row['title']}")
        if 'summary' in row:
            summary = row['summary'][:500] + '...' if len(str(row['summary'])) > 500 else row['summary']
            print(f"   Summary: {summary}")
        print()

---

# Part 5: Query the Knowledge Graph

## 5.1 Two Types of Queries

GraphRAG supports two query modes:

| Mode | Best For | How It Works |
|------|----------|-------------|
| **Local** | Specific questions about entities | Finds relevant entities, traverses relationships |
| **Global** | Big-picture questions | Uses community summaries |

### Examples:

**Local Query**: "What does AuthService depend on?"
- Finds the AuthService entity
- Follows DEPENDS_ON relationships
- Returns: PostgreSQL, Redis, Azure Key Vault

**Global Query**: "Summarize the system architecture"
- Looks at all community summaries
- Synthesizes a high-level answer
- Returns: Overview of services, teams, and dependencies

## 5.2 Local Queries (Entity-Centric)

In [ ]:
# Change to graphrag directory for queries
os.chdir(GRAPHRAG_ROOT)
print(f"📁 Working directory: {os.getcwd()}")

In [ ]:
def run_graphrag_query(query: str, method: str = "local") -> str:
    """
    Run a GraphRAG query using the CLI.
    
    Args:
        query: The question to ask
        method: 'local' for entity-centric, 'global' for community-based
    
    Returns:
        The answer from GraphRAG
    """
    try:
        result = subprocess.run(
            ["graphrag", "query", "--root", ".", "--method", method, "--query", query],
            capture_output=True,
            text=True,
            timeout=120
        )
        
        if result.returncode == 0:
            return result.stdout
        else:
            return f"Error: {result.stderr}"
            
    except subprocess.TimeoutExpired:
        return "Query timed out after 2 minutes"
    except Exception as e:
        return f"Error: {str(e)}"

print("✅ Query function defined")

In [ ]:
# Local Query 1: Dependencies
query1 = "What does AuthService depend on?"

print("🔍 LOCAL QUERY 1")
print("=" * 60)
print(f"Question: {query1}")
print("\nAnswer:")
print("-" * 60)

answer1 = run_graphrag_query(query1, method="local")
print(answer1)

In [ ]:
# Local Query 2: Impact Analysis
query2 = "What happens if AuthService goes down?"

print("🔍 LOCAL QUERY 2")
print("=" * 60)
print(f"Question: {query2}")
print("\nAnswer:")
print("-" * 60)

answer2 = run_graphrag_query(query2, method="local")
print(answer2)

In [ ]:
# Local Query 3: Team Ownership
query3 = "Who owns the API Gateway?"

print("🔍 LOCAL QUERY 3")
print("=" * 60)
print(f"Question: {query3}")
print("\nAnswer:")
print("-" * 60)

answer3 = run_graphrag_query(query3, method="local")
print(answer3)

## 5.3 Global Queries (Community-Based)

In [ ]:
# Global Query 1: System Overview
query4 = "Summarize the overall system architecture"

print("🌍 GLOBAL QUERY 1")
print("=" * 60)
print(f"Question: {query4}")
print("\nAnswer:")
print("-" * 60)

answer4 = run_graphrag_query(query4, method="global")
print(answer4)

In [ ]:
# Global Query 2: Team Structure
query5 = "What are all the teams and what do they own?"

print("🌍 GLOBAL QUERY 2")
print("=" * 60)
print(f"Question: {query5}")
print("\nAnswer:")
print("-" * 60)

answer5 = run_graphrag_query(query5, method="global")
print(answer5)

In [ ]:
# Global Query 3: Incident Lessons
query6 = "What lessons were learned from past incidents?"

print("🌍 GLOBAL QUERY 3")
print("=" * 60)
print(f"Question: {query6}")
print("\nAnswer:")
print("-" * 60)

answer6 = run_graphrag_query(query6, method="global")
print(answer6)

In [ ]:
# Return to original directory
os.chdir(original_dir)
print(f"📁 Returned to: {os.getcwd()}")

---

# Part 6: Compare GraphRAG vs Regular RAG

## 6.1 The Same Questions, Different Approaches

Let's compare how **Regular RAG** (Module 5) and **GraphRAG** answer the same questions.

We'll simulate Regular RAG using simple vector search on our documents.

In [ ]:
from openai import AzureOpenAI

# Initialize Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-12-01-preview"
)

print("✅ Azure OpenAI client initialized")

In [ ]:
def simple_rag_answer(question: str, documents: list[str]) -> str:
    """
    Simulate simple RAG: concatenate all docs and ask the LLM.
    This is what Module 5 does (but with vector search for relevant chunks).
    """
    # Combine all documents as context
    context = "\n\n---\n\n".join(documents)
    
    # Truncate if too long
    if len(context) > 15000:
        context = context[:15000] + "\n\n[...truncated...]"
    
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer questions based ONLY on the provided context. If you can't find the answer in the context, say so."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ],
        temperature=0.0,
        max_tokens=1000
    )
    
    return response.choices[0].message.content

print("✅ Simple RAG function defined")

In [ ]:
# Load our documents for simple RAG comparison
documents = []
for f in sorted(INPUT_DIR.glob("*.txt")):
    documents.append(f.read_text())

print(f"📄 Loaded {len(documents)} documents for comparison")

In [ ]:
# Comparison Question 1: Dependencies (GraphRAG should excel)
comparison_q1 = "What happens if AuthService goes down? List all affected services."

print("📊 COMPARISON: Dependency Question")
print("=" * 70)
print(f"Question: {comparison_q1}")
print()

# Simple RAG Answer
print("🔹 SIMPLE RAG Answer:")
print("-" * 70)
simple_answer = simple_rag_answer(comparison_q1, documents)
print(simple_answer)
print()

In [ ]:
# GraphRAG Answer for the same question
os.chdir(GRAPHRAG_ROOT)

print("🔸 GRAPHRAG Answer:")
print("-" * 70)
graphrag_answer = run_graphrag_query(comparison_q1, method="local")
print(graphrag_answer)

os.chdir(original_dir)

In [ ]:
# Comparison Question 2: Summarization (GraphRAG should excel)
comparison_q2 = "Provide a comprehensive overview of the platform's team structure and service ownership."

print("📊 COMPARISON: Summarization Question")
print("=" * 70)
print(f"Question: {comparison_q2}")
print()

# Simple RAG Answer
print("🔹 SIMPLE RAG Answer:")
print("-" * 70)
simple_answer2 = simple_rag_answer(comparison_q2, documents)
print(simple_answer2)
print()

In [ ]:
# GraphRAG Answer
os.chdir(GRAPHRAG_ROOT)

print("🔸 GRAPHRAG Answer:")
print("-" * 70)
graphrag_answer2 = run_graphrag_query(comparison_q2, method="global")
print(graphrag_answer2)

os.chdir(original_dir)

## 6.2 When to Use Which?

Based on our experiments, here's a decision framework:

| Question Type | Regular RAG | GraphRAG | Winner |
|---------------|-------------|----------|--------|
| "What is X?" (simple fact) | ✅ Good | ✅ Good | **Tie** (RAG is cheaper) |
| "What depends on X?" | ⚠️ Partial | ✅ Excellent | **GraphRAG** |
| "What happens if X fails?" | ⚠️ Partial | ✅ Excellent | **GraphRAG** |
| "Summarize everything about Y" | ⚠️ Limited | ✅ Excellent | **GraphRAG** |
| "Who is responsible for X?" | ✅ Good | ✅ Good | **Tie** |
| "List all components that..." | ❌ Poor | ✅ Excellent | **GraphRAG** |

### Cost Comparison

| Aspect | Regular RAG | GraphRAG |
|--------|-------------|----------|
| **Indexing Cost** | Low (embeddings only) | **HIGH** (many LLM calls) |
| **Query Cost** | Low (1 LLM call) | Medium (1-2 LLM calls) |
| **Query Latency** | Fast (~1 sec) | Slower (~3-10 sec) |
| **Storage** | Vector index | Graph + summaries |

### Recommendation

```
Use REGULAR RAG (Module 5) when:
├── Simple fact-based questions
├── Cost is a concern
├── Real-time responses needed
└── Single document lookups

Use GRAPHRAG when:
├── Cross-document reasoning
├── Dependency/impact analysis
├── System-wide summarization
└── "Connect the dots" questions

Use BOTH (Hybrid) when:
├── Mixed question types
└── Production systems with diverse queries
```

---

# Part 7: Build a Hybrid RAG + GraphRAG Pipeline

## 7.1 Query Router

In production, you'd want to **automatically route** questions to the right system.

Here's a simple implementation:

In [ ]:
def classify_query(question: str) -> str:
    """
    Use an LLM to classify whether a question needs Regular RAG or GraphRAG.
    
    Returns: 'rag', 'graphrag_local', or 'graphrag_global'
    """
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "system",
                "content": """You are a query classifier. Classify questions into one of three categories:

1. 'rag' - Simple fact lookups, specific information retrieval
   Examples: "What is the voltage of X?", "When was Y released?", "What is the price?"

2. 'graphrag_local' - Questions about relationships, dependencies, or impact
   Examples: "What depends on X?", "What happens if Y fails?", "Who owns Z?"

3. 'graphrag_global' - Questions requiring summarization or global view
   Examples: "Summarize the architecture", "List all teams", "What are the main components?"

Respond with ONLY one word: 'rag', 'graphrag_local', or 'graphrag_global'"""
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0.0,
        max_tokens=20
    )
    
    classification = response.choices[0].message.content.strip().lower()
    
    # Validate response
    if classification not in ['rag', 'graphrag_local', 'graphrag_global']:
        return 'rag'  # Default to regular RAG
    
    return classification

print("✅ Query classifier defined")

In [ ]:
def hybrid_answer(question: str) -> dict:
    """
    Smart query router that picks the best approach for each question.
    
    Returns: dict with 'method', 'answer', and 'reasoning'
    """
    # Step 1: Classify the question
    method = classify_query(question)
    
    # Step 2: Route to appropriate system
    if method == 'rag':
        answer = simple_rag_answer(question, documents)
        reasoning = "Simple fact lookup - using Regular RAG"
    elif method == 'graphrag_local':
        os.chdir(GRAPHRAG_ROOT)
        answer = run_graphrag_query(question, method="local")
        os.chdir(original_dir)
        reasoning = "Relationship/dependency question - using GraphRAG Local"
    else:  # graphrag_global
        os.chdir(GRAPHRAG_ROOT)
        answer = run_graphrag_query(question, method="global")
        os.chdir(original_dir)
        reasoning = "Summarization/global question - using GraphRAG Global"
    
    return {
        'method': method,
        'reasoning': reasoning,
        'answer': answer
    }

print("✅ Hybrid answer function defined")

In [ ]:
# Test the hybrid system
test_questions = [
    "What is the p99 latency requirement for API Gateway?",
    "What happens if PostgreSQL becomes unavailable?",
    "Summarize all the services and their owners",
    "Who should I contact about AuthService issues?",
    "What were the action items from the January incident?"
]

print("🤖 HYBRID RAG SYSTEM TEST")
print("=" * 70)

for i, q in enumerate(test_questions, 1):
    print(f"\n📝 Question {i}: {q}")
    print("-" * 70)
    
    result = hybrid_answer(q)
    
    print(f"🎯 Method: {result['method']}")
    print(f"💡 Reasoning: {result['reasoning']}")
    print(f"\n📖 Answer:")
    print(result['answer'][:500] + "..." if len(result['answer']) > 500 else result['answer'])
    print()

---

# Part 8: Summary & Key Takeaways

## 🎓 What We Learned

### 1. **GraphRAG Builds Knowledge Graphs**
```
Documents → Entities → Relationships → Graph → Communities → Summaries
```

### 2. **Two Query Modes**
- **Local**: Entity-centric, traverses relationships
- **Global**: Community-based, uses summaries

### 3. **GraphRAG Excels At**
- Dependency analysis ("What depends on X?")
- Impact analysis ("What if X fails?")
- Cross-document reasoning
- Global summarization

### 4. **GraphRAG Costs More**
- Indexing requires many LLM calls
- Queries are slower than vector search
- Worth it for relationship-heavy domains

### 5. **Best Practice: Hybrid Approach**
- Route simple questions to Regular RAG
- Route relationship questions to GraphRAG
- Use a classifier to decide automatically

---

## 🔧 Microsoft Technologies Used

| Technology | Role |
|------------|------|
| **Microsoft GraphRAG** | Open-source library for graph-based RAG |
| **Azure OpenAI GPT-4.1** | Entity/relationship extraction + generation |
| **Azure OpenAI text-embedding-3-large** | Vector embeddings |

### What We Did NOT Use (in this module):
- ❌ Azure AI Search (GraphRAG has its own retrieval)
- ❌ Azure Document Intelligence (we used pre-extracted text)
- ❌ Azure Content Understanding (same reason)
- ❌ Azure Cosmos DB (GraphRAG uses local files)

---

## 🎯 When to Use GraphRAG in Production

```
✅ USE GRAPHRAG:
├── Architecture documentation
├── Incident investigation ("what caused this?")
├── Compliance traceability
├── Dependency mapping
└── Knowledge base summarization

❌ DON'T USE GRAPHRAG:
├── Simple FAQ systems
├── Real-time chatbots (too slow)
├── Frequently updated content (reindex cost)
└── Single-document Q&A
```

---

# 🎉 Congratulations!

You've completed the RAG Workshop!

## Your Journey:

```
Module 0: Setup              → Got Azure resources ready
Module 1: Naive RAG          → Saw why simple approaches fail
Module 2: Doc Intelligence   → Learned to extract structure
Module 3: Content Under.     → Got AI-powered descriptions
Module 4: Chunking           → Mastered chunking strategies
Module 5: Search             → Built vector + hybrid search
Module 6: GraphRAG           → Added cross-document reasoning ✅
```

## You Can Now Build:

- **Production RAG pipelines** for technical documents
- **Multimodal retrieval** with tables, figures, and charts
- **Hybrid search** (BM25 + vector + semantic reranking)
- **GraphRAG systems** for relationship-heavy domains
- **Query routers** that pick the best approach automatically

## Next Steps:

1. Try GraphRAG on YOUR documents
2. Customize entity types for your domain
3. Build a hybrid RAG + GraphRAG API
4. Add evaluation metrics (Module 5 has groundedness, relevance)
5. Deploy to Azure with proper monitoring

---

**Questions?** Reach out to your workshop facilitator.

**Happy building!** 🚀